In [32]:
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet


In [33]:
game_env = make("orbit_wars", debug=True)
game_env.reset(num_agents=2)
game_env

In [34]:
game_env.state

[{'action': [],
  'reward': 0,
  'info': {},
  'observation': {'remainingOverageTime': 2,
   'step': 0,
   'planets': [],
   'fleets': [],
   'player': 0,
   'angular_velocity': 0,
   'initial_planets': [],
   'next_fleet_id': 0,
   'comets': [],
   'comet_planet_ids': []},
  'status': 'ACTIVE'},
 {'action': [],
  'reward': 0,
  'info': {},
  'observation': {'remainingOverageTime': 2,
   'planets': [],
   'fleets': [],
   'player': 0,
   'angular_velocity': 0,
   'initial_planets': [],
   'next_fleet_id': 0,
   'comets': [],
   'comet_planet_ids': []},
  'status': 'ACTIVE'}]

In [35]:
len(game_env.steps)

1

In [36]:
new_state = game_env.step([[], []])

In [37]:
observation = new_state[0]["observation"]

In [38]:
observation["planets"][0]

[0, -1, 57.19438295977932, 98.49299600496491, 1.0, 50, 1]

In [39]:
planet_0 = Planet(*observation["planets"][0])

In [40]:
planet_0

Planet(id=0, owner=-1, x=57.19438295977932, y=98.49299600496491, radius=1.0, ships=50, production=1)

In [41]:
import sys, math
sys.path.insert(0, "/home/doruk/Documents/Kaggle/Orbit_Wars")

# Reload-friendly import (so edits to feature_extraction.py pick up without restart)
import importlib, feature_extraction
importlib.reload(feature_extraction)
from feature_extraction import Planet as MyPlanet, parse_planets

# --- 1) parse_planets: full observation -> list[MyPlanet] ----------------
planets = parse_planets(observation)
print(f"Parsed {len(planets)} planets")
print(f"  rotating: {sum(p.is_rotating for p in planets)}")
print(f"  static  : {sum(p.is_static   for p in planets)}")
print(f"  comets  : {sum(p.is_comet    for p in planets)}")
print()

# --- 2) Show a few representative planets --------------------------------
for p in planets[:5]:
    print(p)
print()

# --- 3) Ownership helpers ------------------------------------------------
me = observation["player"]
print(f"Player {me}:")
print(f"  mine    : {[p.id for p in planets if p.is_mine(me)]}")
print(f"  enemy   : {[p.id for p in planets if p.is_enemy(me)]}")
print(f"  neutral : {len([p for p in planets if p.is_neutral()])} planets")
print()

# --- 4) Position prediction: pick a rotating planet, project 50 turns ---
omega = observation["angular_velocity"]
rot = next((p for p in planets if p.is_rotating), None)
if rot is not None:
    print(f"Rotating planet id={rot.id}, omega={omega:.4f} rad/turn")
    print(f"  now (t= 0): ({rot.x:6.2f}, {rot.y:6.2f})  theta={math.degrees(rot.theta):+7.1f}°")
    for k in (10, 50, 100, 250):
        fx, fy = rot.position_at(k, angular_velocity=omega)
        print(f"  t=+{k:3d}  : ({fx:6.2f}, {fy:6.2f})")
else:
    print("No rotating planet in this game (rare).")
print()

# --- 5) Static planet sanity: position_at must be invariant -------------
stat = next((p for p in planets if p.is_static), None)
if stat is not None:
    fx, fy = stat.position_at(100, angular_velocity=omega)
    assert (fx, fy) == (stat.x, stat.y), "Static planet shouldn't move!"
    print(f"Static planet id={stat.id} stays at ({stat.x:.2f}, {stat.y:.2f}) ✓")

# --- 6) ships_at projection ---------------------------------------------
print()
sample = planets[0]
print(f"{sample}")
print(f"  ships now           : {sample.ships}")
print(f"  ships in 20 turns   : {sample.ships_at(20):.1f}  "
      f"(neutral=no production, owned=+{sample.production}/turn)")

Parsed 28 planets
  rotating: 8
  static  : 20
  comets  : 0

Planet(id=0, owner=-1, static, r=49.0, theta=+82deg, ships=50, prod=1)
Planet(id=1, owner=-1, static, r=49.0, theta=+98deg, ships=50, prod=1)
Planet(id=2, owner=-1, static, r=49.0, theta=-82deg, ships=50, prod=1)
Planet(id=3, owner=-1, static, r=49.0, theta=-98deg, ships=50, prod=1)
Planet(id=4, owner=-1, static, r=48.9, theta=+19deg, ships=35, prod=3)

Player 0:
  mine    : [12]
  enemy   : [15]
  neutral : 26 planets

Rotating planet id=12, omega=0.0310 rad/turn
  now (t= 0): ( 71.23,  71.23)  theta=  +45.0°
  t=+ 10  : ( 63.76,  76.69)
  t=+ 50  : ( 29.26,  71.72)
  t=+100  : ( 27.81,  29.77)
  t=+250  : ( 31.36,  73.54)

Static planet id=0 stays at (57.19, 98.49) ✓

Planet(id=0, owner=-1, static, r=49.0, theta=+82deg, ships=50, prod=1)
  ships now           : 50
  ships in 20 turns   : 50.0  (neutral=no production, owned=+1/turn)


In [42]:
import importlib, navigation
importlib.reload(navigation)
from navigation import (
    fleet_speed, crosses_sun, find_intercepts,
    path_collision, compute_launch, to_move,
)

omega = observation["angular_velocity"]
me = observation["player"]

# 1) Speed sanity ---------------------------------------------------------
print("Fleet speed examples:")
for n in (1, 5, 50, 200, 500, 1000, 2000):
    print(f"  {n:>4d} ships -> {fleet_speed(n):.3f} units/turn")
print()

# 2) Pick two planets to test --------------------------------------------
my_planets = [p for p in planets if p.is_mine(me)]
src = my_planets[0]                     # our home

# Find a representative target: prefer rotating (more interesting test)
rot_targets = [p for p in planets if p.is_rotating and p.id != src.id]
stat_targets = [p for p in planets if p.is_static and p.id != src.id]
target = rot_targets[0] if rot_targets else stat_targets[0]

print(f"src    : {src}")
print(f"target : {target}")
print()

# 3) Direct angle (no lead) vs intercept -----------------------------------
ships = 50
naive_angle = math.atan2(target.y - src.y, target.x - src.x)
naive_dist = math.hypot(target.x - src.x, target.y - src.y)
print(f"Naive (no lead): angle={math.degrees(naive_angle):+7.2f}°, "
      f"dist={naive_dist:.2f}, ETA={naive_dist/fleet_speed(ships):.2f}")

ints = find_intercepts(src, target, ships, omega, max_results=4)
print(f"\nIntercept candidates ({len(ints)}):")
for k, (t_star, ix, iy) in enumerate(ints):
    a = math.degrees(math.atan2(iy - src.y, ix - src.x))
    print(f"  #{k}: t*={t_star:7.3f}, intercept=({ix:6.2f},{iy:6.2f}), angle={a:+7.2f}°")

# 4) Sun crossing on first candidate? -------------------------------------
if ints:
    t_star, ix, iy = ints[0]
    angle0 = math.atan2(iy - src.y, ix - src.x)
    sx = src.x + math.cos(angle0) * (src.radius + 0.1)
    sy = src.y + math.sin(angle0) * (src.radius + 0.1)
    print(f"\nFirst candidate crosses sun? {crosses_sun(sx, sy, ix, iy)}")

# 5) Top-level compute_launch --------------------------------------------
result = compute_launch(src, target, ships, planets, omega)
print(f"\ncompute_launch result:")
if result is None:
    print("  None  (no valid trajectory; sun and/or collisions block all candidates)")
else:
    print(f"  angle           : {math.degrees(result['angle']):+.3f}°")
    print(f"  ships           : {result['ships']}")
    print(f"  eta             : {result['eta']:.3f} turns")
    print(f"  intercept       : ({result['intercept_x']:.2f}, {result['intercept_y']:.2f})")
    print(f"  candidate_index : {result['candidate_index']}  (0 = soonest)")
    print(f"  env move        : {to_move(result, src)}")

# 6) Stress: try every (mine -> any other) pair, count successes ---------
print("\nLaunch feasibility for ships=50, every (mine -> other) pair:")
ok = blocked = 0
for s in my_planets:
    for t in planets:
        if t.id == s.id:
            continue
        r = compute_launch(s, t, 50, planets, omega)
        if r is None:
            blocked += 1
        else:
            ok += 1
print(f"  feasible: {ok}, blocked: {blocked}")

Fleet speed examples:
     1 ships -> 1.000 units/turn
     5 ships -> 1.562 units/turn
    50 ships -> 3.131 units/turn
   200 ships -> 4.359 units/turn
   500 ships -> 5.267 units/turn
  1000 ships -> 6.000 units/turn
  2000 ships -> 6.771 units/turn

src    : Planet(id=12, owner=0, rot, r=30.0, theta=+45deg, ships=10, prod=5)
target : Planet(id=13, owner=-1, rot, r=30.0, theta=+135deg, ships=19, prod=5)

Naive (no lead): angle=+180.00°, dist=42.47, ETA=13.56

Intercept candidates (1):
  #0: t*= 16.559, intercept=( 21.08, 58.09), angle=-165.32°

First candidate crosses sun? False

compute_launch result:
  angle           : -165.316°
  ships           : 50
  eta             : 16.559 turns
  intercept       : (21.08, 58.09)
  candidate_index : 0  (0 = soonest)
  env move        : [12, -2.8853134805041503, 50]

Launch feasibility for ships=50, every (mine -> other) pair:
  feasible: 17, blocked: 10
